# URL feature engineering and Decision Tree

Madushan S.C · URLSentra · Proposal §§4.6–4.10 and 4.12 (integrated with the group's 2026-11-24 methodology decisions)

Place the PhiUSIIL CSV in `data/raw/` and run from the repository root or `notebooks/`. The default run develops on a 20,000-row sample. For the final experiment set `SAMPLE = None` and enable `RUN_FINAL_TEST` only after the feature sets and the tree configuration are frozen.

Phishing is class 1 after the shared target conversion, and every precision, recall and F1 below refers to phishing.

Protocol used here (same as `main.py`):
1. Near-duplicate URLs form one **URL group**; the train/test split and every cross-validation keep a group on one side.
2. Feature sets are chosen on **training rows only**: `url_base`, `url_engineered` (primary), `extended` (secondary). `URLSimilarityIndex` is excluded from all of them (label proxy).
3. The tree is tuned with **grouped CV on training rows**; the held-out test set is used once, at the end, with the frozen configuration.

**Status:** this notebook is committed without outputs. Any number not printed by a cell you have run yourself is not verified by this notebook; the experiment log below states where each logged number came from.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from src import config, data, evaluate, features
from src.models import tree

SAMPLE = 20_000
RUN_FINAL_TEST = False

df = data.load_clean_data()
if SAMPLE is not None and SAMPLE < len(df):
    df = df.sample(n=SAMPLE, random_state=config.RANDOM_STATE).reset_index(drop=True)

original = data.get_original_features(df)
features.check_feature_groups(original)  # every column is classed as URL or page content
df = features.add_features(df)

# One split for every feature set: rows depend only on the URLs and labels.
groups = data.group_ids(df)
candidates = original + features.ENGINEERED_FEATURES
X_train, X_test, y_train, y_test = data.split(df, candidates)
train_groups = groups.loc[X_train.index]
print(f"Training rows: {len(X_train):,} | Holdout rows: {len(X_test):,}")

## URL groups and the group-aware split

`data.url_group_key()` normalizes a URL only for grouping (the `URL` column and the features are computed from the original string):

| Step | Example |
|---|---|
| Remove `http://` / `https://` | `https://a.com/x` → `a.com/x` |
| Lower-case the hostname | `A.COM/x` → `a.com/x` |
| Remove one leading `www.` from the hostname | `www.a.com` → `a.com` |
| Remove trailing `/` from the path | `a.com/x/` → `a.com/x` |
| **Kept as written** | path case, query string (case and content), fragment, port, user info |

Path and query case are kept because they can address different pages; dropping the query merged unrelated URLs into groups of up to 45 rows.

`data.split()` uses `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)` and takes one fold (about 20 %) as the held-out test set. Groups whose rows carry **different labels** are kept unchanged and reported as dataset label noise; because a group never crosses the split, the conflict cannot leak into the test set.

In [ ]:
sizes = groups.value_counts()
conflicts = data.conflicting_label_groups(df)
shared = set(train_groups) & set(groups.loc[X_test.index])
display(pd.Series({
    "rows": len(df),
    "URL groups": groups.nunique(),
    "groups with more than one row": int((sizes > 1).sum()),
    "largest group": int(sizes.max()),
    "label-conflict groups (kept)": conflicts["url_group_key"].nunique(),
    "groups in both train and test": len(shared),
    "test share": round(len(X_test) / len(df), 4),
    "phishing share, train": round(y_train.mean(), 4),
    "phishing share, test": round(y_test.mean(), 4),
}).to_frame("value"))
assert not shared, "a URL group crosses the train/test split"
display(conflicts.head(10))

## Feature sets and engineered URL signals

The shared lists live in `src/features.py` (`URL_FEATURES`, `PAGE_FEATURES`, `EXCLUDED_FEATURES`, `COLLECTION_BIAS_FEATURES`); they were moved there from notebook 04, and `check_feature_groups()` confirms they cover the dataset's numeric columns exactly.

| Set | Contents | Role |
|---|---|---|
| `url_base` | URL-string features minus `URLSimilarityIndex` and `IsHTTPS` | baseline |
| `url_engineered` | `url_base` + non-redundant engineered features | **primary** |
| `extended` | `url_engineered` + `IsHTTPS` + page-content features | secondary comparison |

Engineered features (row-wise, from the URL string only, so computing them before the split leaks nothing):

| Signal | What it measures |
|---|---|
| Path depth | Number of non-empty path segments |
| Longest path segment | Largest segment length, which can indicate large identifiers or slugs |
| Query field count | Number of non-empty ampersand-separated query fields |
| Encoded byte count | Number of valid percent-encoded bytes in the path and query |

`features.build_feature_sets()` runs on the training rows only. It removes constant columns and, of any pair with absolute Pearson correlation ≥ 0.98, keeps the column listed first (existing columns come first). This only removes near-copies; passing the filter does **not** show that a feature helps. Whether the engineered features help is judged by the grouped CV comparison of `url_base` and `url_engineered` below. Correlation also misses nonlinear redundancy.

In [ ]:
feature_sets = features.build_feature_sets(df.loc[X_train.index])  # training rows only
primary = features.PRIMARY_FEATURE_SET
listed = features.url_base_candidates() + features.ENGINEERED_FEATURES + list(features.COLLECTION_BIAS_FEATURES) + features.PAGE_FEATURES
display(pd.DataFrame({
    "feature": listed,
    "engineered": [c in features.ENGINEERED_FEATURES for c in listed],
    **{name: [c in cols for c in listed] for name, cols in feature_sets.items()},
}))
print({name: len(cols) for name, cols in feature_sets.items()})
removed = [c for c in features.ENGINEERED_FEATURES if c not in feature_sets[primary]]
print("Engineered features removed as redundant on training rows:", removed)

# Strongest absolute correlation of each engineered feature with a url_base column (training rows).
corr = X_train[feature_sets["url_base"] + features.ENGINEERED_FEATURES].corr().abs()
display(pd.DataFrame({
    e: corr.loc[feature_sets["url_base"], e].agg(["max", "idxmax"]) for e in features.ENGINEERED_FEATURES
}).T)
display(X_train[features.ENGINEERED_FEATURES].describe().round(2))

## Training-only grouped tuning

`tree.tune(X, y, groups=...)` grid-searches depth, minimum leaf size and class weighting with grouped 5-fold CV on the training rows of the **primary** set. The folds come from `evaluate.cv_folds()`, so they are identical to the folds used for the other models on the same rows. Phishing F1 picks the winner; precision, recall, accuracy and ROC-AUC are recorded for every candidate.

The configuration `main.py` evaluates is frozen in `tree.build()` (`BEST_*` constants). It was chosen on the **full** training split by `tree.tune()` plus the complexity review below (`tree.TUNING_NOTE`). On the 20,000-row sample the search may pick something else; the cell below says so explicitly. Final results always use the frozen `tree.build()`, so this notebook and `main.py` report the same model. These CV scores are development results, not test results.

In [ ]:
search = tree.tune(X_train[feature_sets[primary]], y_train, groups=train_groups)
cv_results = pd.DataFrame(search.cv_results_)
show = ["rank_test_f1", "params"] + [f"mean_test_{m}" for m in ("f1", "precision", "recall", "roc_auc")] + ["std_test_f1"]
display(cv_results[show].sort_values("rank_test_f1").head().round(4))

frozen = {k: tree.build().get_params()[k] for k in tree.PARAM_GRID}
print("Best on this run:", search.best_params_)
print("Frozen tree.build():", frozen, "|", tree.TUNING_NOTE)
if tree.is_frozen_config(search.best_params_):
    print("This run agrees with the frozen configuration.")
else:
    print("This run (SAMPLE =", SAMPLE, ") prefers a different setting. Final results still use the frozen "
          "tree.build(), chosen on the full training split; change it only after a full-data tune().")

# Grouped CV of the frozen tree on all three sets (identical folds; training rows only).
cv_sets = evaluate.compare_training_models(
    {name: (X_train[cols], y_train) for name, cols in feature_sets.items()},
    {"Decision Tree": tree.build()},
    groups=train_groups,
)
display(cv_sets.round(4))
display(evaluate.feature_delta_table(cv_sets, base="url_base", other="url_engineered").round(4))


## Complexity review: why `max_depth = 20`

The first full-data grid winner was the unlimited tree (`max_depth=None`, leaf 1). A tree is only worth its
extra size if it buys real CV performance, so `tune_models.py` (part `dt`) also measured the size of every
fitted fold tree and ran a depth curve (8 … 32, leaf 1). Rule fixed **before** the run (CART one-standard-error
rule, Breiman et al. 1984; Hastie et al., *ESL* 7.10): among all evaluated configurations, take the one with the
**fewest leaves** whose mean CV F1 ≥ best mean − SE(best), SE = sample std of the best's five fold scores / √5.

The cell below reads the stored full-data table (`reports/tuning/dt_complexity.csv`, grouped 5-fold CV on all
188,297 training rows, `url_engineered`); it does not rerun the search (about 3 minutes on the full split).


In [ ]:

comp_path = config.RESULTS_DIR / "tuning" / "dt_complexity.csv"
comp = pd.read_csv(comp_path)
cols = ["source", "max_depth", "min_samples_leaf", "class_weight", "f1_mean", "f1_std", "f1_se",
        "roc_auc_mean", "depth_mean", "leaves_mean", "nodes_mean", "within_1se_of_best"]
display(comp[cols].head(12).round(5))

best = comp.iloc[0]
chosen = comp[comp["within_1se_of_best"]].sort_values(["leaves_mean", "f1_mean"], ascending=[True, False]).iloc[0]
print(f"best mean F1 {best.f1_mean:.5f} (SE {best.f1_se:.5f}) -> threshold {best.f1_mean - best.f1_se:.5f}")
print(f"one-SE choice: max_depth={chosen.max_depth}, leaf={chosen.min_samples_leaf}, "
      f"F1 {chosen.f1_mean:.5f}, {chosen.leaves_mean:.0f} leaves, depth {chosen.depth_mean:.0f}")
assert (tree.BEST_MAX_DEPTH, tree.BEST_MIN_SAMPLES_LEAF) == (int(chosen.max_depth), int(chosen.min_samples_leaf)), \
    "tree.build() does not match the stored complexity review"

# F1 against tree size (leaf 1, no class weight): where extra depth stops paying.
curve = comp[(comp["min_samples_leaf"] == 1) & comp["class_weight"].isna()].copy()
curve["depth_label"] = curve["max_depth"].map(lambda d: "None" if pd.isna(d) else str(int(d)))
curve = curve.sort_values("leaves_mean")
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.errorbar(curve["leaves_mean"], curve["f1_mean"], yerr=curve["f1_std"], marker="o", capsize=3)
labels = {"12": ("depth 12", (6, -4)), "16": ("depth 16", (6, -10)), "20": ("depth 20 (chosen)", (-30, 8)),
          "None": (f"unlimited (depth {comp.loc[comp['max_depth'].isna() & (comp['min_samples_leaf'] == 1) & comp['class_weight'].isna(), 'depth_mean'].iloc[0]:.0f})", (-60, -16))}
for r in curve.itertuples():
    if r.depth_label in labels:
        text, offset = labels[r.depth_label]
        ax.annotate(text, (r.leaves_mean, r.f1_mean), textcoords="offset points", xytext=offset, fontsize=8)
ax.axhline(best.f1_mean - best.f1_se, color="grey", ls="--", lw=1, label="best − 1 SE")
ax.set(xlabel="mean leaves per fold tree (depth 24, 28, 32 between depth 20 and unlimited)",
       ylabel="grouped CV F1 (phishing)", title="Decision Tree: CV F1 vs size (full training split, url_engineered)")
ax.set_xlim(550, 1650)
ax.set_ylim(0.95, 0.993)
ax.legend(loc="lower right")
fig.savefig(config.FIGURES_DIR / "dt_complexity_curve.png", dpi=150, bbox_inches="tight")
plt.show()



**Decision: `max_depth = 20`, `min_samples_leaf = 1`, no class weight** (frozen in `tree.build()`, used by
`main.py`, this notebook and the final evaluation).

- The unlimited tree grows to depth 37 and about 1,564 leaves; depth 20 has about 1,299 leaves (−17 %) and its
  mean CV F1 is not lower (0.99053 vs 0.99008, a difference of about 1.5 fold standard deviations, i.e. "as good
  and simpler", not "better").
- Depth 16 saves another ~270 leaves but loses about 0.003 F1 (outside one SE); depth 12 or less loses 0.04 or more.
- **Interpretability:** no tree in this range can be read leaf by leaf (a depth-4 tree with 15 leaves loses about
  0.17 F1). For the demonstration the tree is explained by its top levels (figure below) and feature importances.
- **Caveat:** depth 20 came from the depth curve, outside the first grid; picking the best of 8 extra
  configurations adds a little optimism to its CV score, which is why the held-out test is reported separately.


In [ ]:

# The top three levels of the frozen tree, fitted on this run's training rows (sample or full).
from sklearn.tree import plot_tree

dt_demo = tree.train(X_train[feature_sets[primary]], y_train)
print(f"fitted tree: depth {dt_demo.get_depth()}, {dt_demo.get_n_leaves():,} leaves (SAMPLE = {SAMPLE})")
fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(dt_demo, max_depth=2, feature_names=feature_sets[primary], class_names=["legitimate", "phishing"],
          filled=True, impurity=False, proportion=True, rounded=True, fontsize=8, ax=ax)
ax.set_title("Decision Tree, top 3 levels (url_engineered); lower levels truncated")
suffix = f"_sample_{SAMPLE}" if SAMPLE else ""
fig.savefig(config.FIGURES_DIR / f"dt_top_levels{suffix}.png", dpi=150, bbox_inches="tight")
plt.show()
display(pd.Series(dt_demo.feature_importances_, index=feature_sets[primary], name="impurity importance")
        .sort_values(ascending=False).head(10).round(4))


## Held-out test comparison

Run this only after the feature sets and the tree configuration are frozen; nothing here feeds back into those choices. All three sets use the same held-out rows and the frozen `tree.build()`, exactly as `main.py` does. `url_engineered` is the primary result; `url_base` and `extended` are controlled comparisons. Check false positives and false negatives as well as F1. Scores near 0.99 or higher call for a check of collection bias and remaining leakage before they are reported.

In [ ]:
if RUN_FINAL_TEST:
    results = {}
    for name, cols in feature_sets.items():
        model = tree.train(X_train[cols], y_train)
        results[(name, "Decision Tree")] = evaluate.evaluate(model, X_test[cols], y_test)
        evaluate.plot_confusion_matrix(model, X_test[cols], y_test, f"Decision Tree: {name} (held-out test)")
    scores = evaluate.compare(results).set_index("feature_set")
    display(scores.round(4))
    metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]
    for base, other in [("url_base", "url_engineered"), ("url_engineered", "extended")]:
        delta = scores.loc[other, metrics] - scores.loc[base, metrics]
        display(delta.astype(float).round(4).rename(f"{other} minus {base}").to_frame())
    plt.show()
else:
    print("Holdout evaluation is disabled until the experiment settings are frozen.")

## Experiment log

| Date | Experiment | Purpose | CV or test | Evidence / status |
|---|---|---|---|---|
| 2026-11-24 | Training CV on original features, ungrouped split (commit 7973c24) | First tree grid | CV | Superseded: ungrouped folds, included page features |
| 2026-11-24 | Grouped `tree.tune()` on full training split, `url_engineered` (23 features) | First grid winner | Grouped CV (training rows only) | Best `max_depth=None, min_samples_leaf=1, class_weight=None`, mean F1 0.99008 (std 0.00030); `reports/tuning/dt_grouped_search.csv` |
| 2026-11-24 | Complexity review + depth curve, one-SE rule (`tune_models.py` part `dt`) | Freeze the simplest tree within noise | Grouped CV (training rows only) | **Frozen: `max_depth=20`, leaf 1**, F1 0.99053 (std 0.00042), 1,299 leaves vs 1,564 unlimited; `reports/tuning/dt_complexity.csv` |
| 2026-11-24 | Per-fold feature selection check (`tune_models.py` part `fs`) | Is selecting columns on all training rows before CV a leak? | Grouped CV folds | All 15 (fold, set) selections equal the training-rows selection; `reports/tuning/feature_selection_folds.csv` |
| 2026-11-24 | `url_base` vs `url_engineered` vs `extended`, frozen tree (full data, `main.py`) | Value of engineered features / page content | Grouped CV | F1 0.9905 / 0.9905 / 0.9988: **the engineered features add nothing measurable** (LR: 0.9972 vs 0.9967, RF 0.9950 vs 0.9950); `extended` is higher only through collection-biased page features; `reports/cv_results.csv` |
| 2026-11-24 | Held-out comparison, frozen tree (full data, `main.py`) | Final report | Test | `url_engineered` F1 0.9917 (TN 26,834 / FP 135 / FN 197 / TP 19,907); format rule 0.9948 — the tree is **below** the format baseline; `reports/results.csv`, `reports/format_baseline.csv` |